# Exploratory Data Analysis: mental-health texts and the Twitter network

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import GX_utils as U

spark = U.get_spark("GX_EDA")
U.setup_storage(spark)
sns.set_theme(style="whitegrid")

mental_health_df = U.read_mental_health_raw(spark)
twitter_graph_df = U.read_twitter_graph_raw(spark)
mental_health_df.printSchema()
twitter_graph_df.printSchema()

## 1. Veracity: missing values per colum

In [0]:
def missing_value_report(df):
    numeric = {"int", "bigint", "double", "float"}
    exprs = []
    for name, dtype in df.dtypes:
        c = F.col(f"`{name}`")
        cond = (c.isNull() | F.isnan(c)) if dtype in numeric else c.isNull()
        exprs.append(F.sum(F.when(cond, 1).otherwise(0)).alias(name))
    total = df.count()
    row = df.select(exprs).first().asDict()
    return spark.createDataFrame([(k, v, round(100.0 * v / total, 2)) for k, v in row.items()],
                                 "column string, missing long, missing_pct double")

missing_value_report(mental_health_df).show(truncate=False)
missing_value_report(twitter_graph_df).show(truncate=False)

## 2. Text: class balance

In [0]:
total_texts = mental_health_df.count()
emotion_distribution = (mental_health_df.groupBy("Emotion").count()
    .withColumn("pct", F.round(F.col("count") / total_texts * 100, 1))
    .orderBy(F.desc("count")))
emotion_distribution.show()

emotion_pd = emotion_distribution.toPandas()
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(emotion_pd["Emotion"], emotion_pd["count"], color=sns.color_palette("mako", len(emotion_pd)))
ax.bar_label(bars, labels=[f"{c:,}\n({p}%)" for c, p in zip(emotion_pd["count"], emotion_pd["pct"])], fontsize=9)
ax.set_title(f"Texts per emotion (N = {total_texts:,})", fontweight="bold")
ax.set_ylabel("Number of texts")
plt.tight_layout(); plt.show()

## 3. Text: length and writing style per emotion

In [0]:
letters = F.length(F.regexp_replace("Text", "[^A-Za-z]", ""))
uppers = F.length(F.regexp_replace("Text", "[^A-Z]", ""))
text_metrics_df = (mental_health_df
    .withColumn("word_count", F.size(F.split(F.trim("Text"), r"\s+")))
    .withColumn("char_length", F.length("Text"))
    .withColumn("upper_ratio", F.when(letters > 0, uppers / letters).otherwise(0.0)))

text_metrics_df.select("word_count", "char_length").summary("mean", "stddev", "min", "25%", "50%", "75%", "max").show()

(text_metrics_df.groupBy("Emotion").agg(
        F.count("*").alias("texts"),
        F.round(F.avg("word_count"), 1).alias("avg_words"),
        F.percentile_approx("word_count", 0.5).alias("median_words"),
        F.max("word_count").alias("max_words"),
        F.round(F.avg((F.col("word_count") > 200).cast("int")) * 100, 1).alias("pct_over_200_words"),
        F.round(F.avg((F.col("upper_ratio") > 0).cast("int")) * 100, 1).alias("pct_with_capitals"))
    .orderBy(F.desc("avg_words"))
    .show())

In [0]:
length_hist = (text_metrics_df
    .withColumn("bucket", F.floor(F.log2(F.col("word_count"))))
    .groupBy("Emotion", "bucket").count()
    .toPandas())
length_hist["share"] = length_hist["count"] / length_hist.groupby("Emotion")["count"].transform("sum")
pivot = length_hist.pivot(index="bucket", columns="Emotion", values="share").fillna(0).sort_index()
pivot.index = [f"{2**int(b)}-{2**(int(b) + 1) - 1}" for b in pivot.index]

ax = pivot.plot(kind="line", marker="o", figsize=(10, 4))
ax.set_title("Text length distribution by emotion", fontweight="bold")
ax.set_xlabel("Words per text (log2 buckets)"); ax.set_ylabel("Share of texts in class")
plt.tight_layout(); plt.show()

Depression posts are long, capitalised Reddit-style posts while most other classes are short lower-case sentences, so length alone could leak the label (tested in `GX_MLlib`).

## 4. Text: top terms per emotion

In [0]:
words_df = (mental_health_df
    .select("Emotion", F.explode(F.split(F.lower("Text"), "[^a-z]+")).alias("word"))
    .filter((F.length("word") > 2) & ~F.col("word").isin(U.EDA_STOPWORDS)))

rank_window = Window.partitionBy("Emotion").orderBy(F.desc("count"))
top_words = (words_df.groupBy("Emotion", "word").count()
    .withColumn("rank", F.row_number().over(rank_window))
    .filter("rank <= 8"))

(top_words.groupBy("Emotion")
    .agg(F.array_join(F.transform(F.sort_array(F.collect_list(F.struct("rank", "word"))), lambda s: s["word"]), ", ")
         .alias("top_8_terms"))
    .orderBy("Emotion")
    .show(truncate=False))

## 5. Network veracity: how much the float-exported Source IDs

In [0]:
id_df = U.maybe_cache(twitter_graph_df.select(
    "Source", "Destination",
    U.normalize_twitter_id(F.col("Source")).alias("src"),
    U.normalize_twitter_id(F.col("Destination")).alias("dst"),
    F.to_timestamp("Relationship Date (UTC)").alias("relationship_ts"),
    F.col("Destination Followers").alias("dst_followers")))

total_edges = id_df.count()
raw_dest = id_df.select(F.col("Destination").alias("Source")).distinct()
fixed_dest = id_df.select(F.col("dst").alias("src")).distinct()

print(f"relationships:                          {total_edges:,}")
print(f"source in scientific notation:          {id_df.filter(U.is_scientific_notation(F.col('Source'))).count():,}")
print(f"source matches a destination (raw):     {id_df.join(raw_dest, 'Source', 'left_semi').count():,}")
print(f"source matches a destination (repaired):{id_df.join(fixed_dest, 'src', 'left_semi').count():,}")
print(f"self-interactions after repair:         {id_df.filter('src = dst').count():,}")

## 6. Network volume: scale and degree distribution

In [0]:
social = id_df.filter("src <> dst")
nodes = social.select(F.col("src").alias("id")).union(social.select(F.col("dst").alias("id"))).distinct()
n_nodes, n_edges = nodes.count(), social.count()
n_pairs = social.select("src", "dst").distinct().count()
print(f"accounts: {n_nodes:,} | interactions: {n_edges:,} | distinct pairs: {n_pairs:,} | density: {n_pairs / (n_nodes * (n_nodes - 1)):.2e}")

out_degree = social.groupBy("src").agg(F.countDistinct("dst").alias("out_degree"))
in_degree = social.groupBy("dst").agg(F.countDistinct("src").alias("in_degree"))
out_degree.select("out_degree").summary("mean", "50%", "90%", "99%", "max").show()
in_degree.select("in_degree").summary("mean", "50%", "90%", "99%", "max").show()

In [0]:
in_freq = in_degree.groupBy("in_degree").count().toPandas()
out_freq = out_degree.groupBy("out_degree").count().toPandas()

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(in_freq["in_degree"], in_freq["count"], s=8, label="in-degree")
ax.scatter(out_freq["out_degree"], out_freq["count"], s=8, label="out-degree")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Degree"); ax.set_ylabel("Number of accounts")
ax.set_title("Degree distribution (log-log)", fontweight="bold")
ax.legend(); plt.tight_layout(); plt.show()

in_degree.orderBy(F.desc("in_degree")).show(10, truncate=False)
twitter_graph_df.groupBy("Source Verified").count().show()

## 7. Velocity: relationships recorded per day

In [0]:
daily = id_df.groupBy(F.to_date("relationship_ts").alias("day")).count().orderBy("day")
daily.select(F.min("day"), F.max("day")).show()

daily_pd = daily.filter(F.col("day") >= "2023-01-01").toPandas()
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(daily_pd["day"], daily_pd["count"], lw=1)
ax.set_title("Relationships recorded per day (2023)", fontweight="bold"); ax.set_ylabel("Relationships")
plt.tight_layout(); plt.show()

## 8. Spark SQL: length percentiles per emotion, longest posts, and the most-engaged accounts per follower tier

In [0]:
text_metrics_df.createOrReplaceTempView("texts")
social.createOrReplaceTempView("interactions")

spark.sql('''
    SELECT Emotion,
           percentile_approx(word_count, 0.5) AS median_words,
           percentile_approx(word_count, 0.9) AS p90_words
    FROM texts GROUP BY Emotion ORDER BY median_words DESC
''').show()

spark.sql('''
    WITH ranked AS (
        SELECT Emotion, word_count, substring(Text, 1, 70) AS snippet,
               ROW_NUMBER() OVER (PARTITION BY Emotion ORDER BY word_count DESC) AS rk
        FROM texts)
    SELECT Emotion, rk, word_count, snippet FROM ranked WHERE rk <= 2 ORDER BY Emotion, rk
''').show(truncate=False)

In [0]:
spark.sql('''
    WITH inbound AS (
        SELECT dst AS account, COUNT(*) AS interactions, COUNT(DISTINCT src) AS distinct_senders,
               MAX(dst_followers) AS followers
        FROM interactions GROUP BY dst),
    tiered AS (
        SELECT *, CASE WHEN followers IS NULL THEN '0 unknown'
                       WHEN followers < 1000 THEN '1 micro (<1K)'
                       WHEN followers < 100000 THEN '2 mid (1K-100K)'
                       ELSE '3 macro (>100K)' END AS tier
        FROM inbound),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY tier ORDER BY distinct_senders DESC) AS rk FROM tiered)
    SELECT tier, rk, account, followers, distinct_senders, interactions
    FROM ranked WHERE rk <= 3 ORDER BY tier, rk
''').show(truncate=False)

**Takeaways:** classes are moderately imbalanced (Positive 23% vs Anxiety 11%); Depression texts are ~6x longer than the rest (median 122 vs 10–23 words); the raw graph is unusable until IDs are repaired; degrees follow a heavy-tailed hub structure.